In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

In [2]:
#load the training and test data 
data_train = pd.read_csv('/home/bigbang/machinelearning_algorithms/multi_classification_train.csv')
X_train = data_train.iloc[:,1:-1].values
y = data_train.iloc[: , -1].values 

data_test = pd.read_csv('/home/bigbang/machinelearning_algorithms/multi_classification_test.csv')
X_test = data_test.iloc[:, 1:].values 

# Normalize training data
X_train_mean = X_train.mean(axis=0)
X_train_std = X_train.std(axis=0)

X = (X_train - X_train_mean) / X_train_std  # Standardize training features
# Add bias term
X = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)

# Normalize test data using training data's mean and std
X_test_normalized = (X_test - X_train_mean) / X_train_std
# Add bias term
X_test_normalized = np.concatenate((np.ones((X_test_normalized.shape[0], 1)), X_test_normalized), axis=1)

print("Test data after normalization:")
print(X_test_normalized[:5])  # Print first 5 rows of normalized test data



# Display the first few rows of the training data to confirm it's loaded correctly
print(X[:5])
print(y[:5])


Test data after normalization:
[[ 1.          0.46426969 -0.03885469  0.80877668  0.06942443 -0.24648566
   1.89374095 -0.72470561  1.26628958  1.35574713 -0.47808792  0.41544472
   0.41544472 -1.18478844 -2.25706795  0.39395261 -0.26120215  0.10680029
   0.12866715  0.38491852 -0.47808792]
 [ 1.         -0.06515093  2.60152516 -0.30475048 -1.29010364 -0.46401948
  -0.81037971 -0.22592233  0.59323918  0.29283471 -0.70017873  0.26057349
   0.26057349  0.74083931  0.61498048  0.92424679  0.23981959 -0.65671046
   0.47049435  1.03163336 -0.70017873]
 [ 1.         -0.47718322  1.28167278 -1.19850683 -1.0779058  -0.29458798
  -1.1154011   1.33130635  0.42728684 -0.60266712  0.76468593  0.64948998
   0.64948998 -0.27825452  0.3129469  -0.44811664  1.72091238  0.76677935
  -0.48886662  0.49932105  0.76468593]
 [ 1.         -1.22599534 -0.80715844  0.3978466   1.68820943 -0.0772888
   1.66064126 -1.13331946  2.19592004 -2.12688135  1.66483866  1.18241224
   1.18241224 -1.41609337  0.642193   -

In [3]:
def softmax(z):
    """
    Compute softmax values for each row of the input matrix z.
    """
    exp_z = np.exp(z) 
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def compute_cost(X, y, w, num_classes):
    """
    Compute the cross-entropy cost for multiclass logistic regression.
    """
    m = X.shape[0]
    logits = np.dot(X, w)
    probs = softmax(logits)
    y_one_hot = np.eye(num_classes)[y]
    cost = -np.sum(y_one_hot * np.log(probs)) / m
    return cost

def gradient_descent(X, y, num_classes, alpha=0.01, epochs=1000):
    """
    Train a multiclass logistic regression model using gradient descent.
    """
    m, n = X.shape
    w = np.zeros((n, num_classes))  # Initialize weights
    
    for epoch in range(epochs):
        logits = np.dot(X, w)
        probs = softmax(logits)
        y_one_hot = np.eye(num_classes)[y]
        gradient = (1 / m) * np.dot(X.T, (probs - y_one_hot))
        w -= alpha * gradient
        
        if epoch % 100 == 0:
            cost = compute_cost(X, y, w, num_classes)
            print(f'Epoch {epoch}, Cost: {cost:.4f}')
    
    return w

def predict(X, w):
    """
    Predict the class labels for input features X.
    
    Parameters:
        X (numpy.ndarray): Feature matrix with shape (m, n).
        w (numpy.ndarray): Trained weight matrix.
    
    Returns:
        numpy.ndarray: Predicted class labels.
    """
    logits = np.dot(X, w)
    probs = softmax(logits)
    return np.argmax(probs, axis=1)





# Train the model
num_classes = len(np.unique(y))
weights = gradient_descent(X, y, num_classes, alpha=0.01, epochs=1000)

# Make predictions
predictions = predict(X, weights)
print("Predictions:", predictions)

# Accuracy
accuracy = np.mean(predictions == y) * 100
print(f"Accuracy: {accuracy:.2f}%")
   
  

Epoch 0, Cost: 1.6019
Epoch 100, Cost: 1.1152
Epoch 200, Cost: 0.9128
Epoch 300, Cost: 0.8099
Epoch 400, Cost: 0.7482
Epoch 500, Cost: 0.7070
Epoch 600, Cost: 0.6774
Epoch 700, Cost: 0.6552
Epoch 800, Cost: 0.6379
Epoch 900, Cost: 0.6240
Predictions: [1 2 4 ... 2 3 2]
Accuracy: 81.18%


In [4]:
predictions_test = predict(X_test_normalized, weights)
print("Prediction of test data:", predictions_test)

Prediction of test data: [3 1 1 ... 3 0 1]


In [5]:
def compute_f1_score(y_true, y_pred, num_classes):
    
    f1_scores = []
    for i in range(num_classes):
        true_positive = np.sum((y_pred == i) & (y_true == i))
        false_positive = np.sum((y_pred == i) & (y_true != i))
        false_negative = np.sum((y_pred != i) & (y_true == i))
        
        precision = true_positive / (true_positive + false_positive + 1e-15)
        recall = true_positive / (true_positive + false_negative + 1e-15)
        f1 = 2 * precision * recall / (precision + recall + 1e-15)
        f1_scores.append(f1)
    
    return np.mean(f1_scores)

f1_score = compute_f1_score(y, predictions, num_classes)
print(f'F1 score : {f1_score}')

F1 score : 0.7547619574385008
